# 10.4 ORM Intro — SQLAlchemy

**Prerequisites:** 10.3 SQLite in Python, 05 OOPs, 5.3 Dataclasses & Enums  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- What an ORM is, and the honest trade-offs against raw SQL
- SQLAlchemy's two layers: Core and ORM
- Declarative models with `Mapped` / `mapped_column` (2.0 style)
- `Engine`, `Session`, and the unit-of-work pattern
- CRUD: `select`, `insert`, `update`, `delete`
- Relationships, and lazy vs eager loading
- 🔴 **The N+1 query problem**
- When an ORM helps, and when it gets in the way

---

## What is an ORM?

An **Object-Relational Mapper** translates between two worlds that do not naturally line up:

| Database | Python |
|---|---|
| Table | Class |
| Row | Instance |
| Column | Attribute |
| Foreign key | A reference to another object |
| `JOIN` | Attribute access |

Instead of writing SQL and unpacking tuples, you work with objects:

```python
# Raw sqlite3 (10.3)
cur.execute("SELECT sku, name, price FROM product WHERE price > ?", (1000,))
for row in cur.fetchall():
    print(row["name"])

# ORM
for product in session.scalars(select(Product).where(Product.price > 1000)):
    print(product.name)
```

### The honest trade-offs

An ORM is not automatically better. It is a trade.

| ORM gives you | ORM costs you |
|---|---|
| Python objects instead of tuples | A second query language to learn |
| Database portability (SQLite → Postgres) | A layer between you and the SQL |
| Automatic parameterisation (injection-safe) | Surprising performance (see **N+1**) |
| Schema as readable class definitions | Harder to express complex queries |
| Change tracking and transactions | Real debugging needs the generated SQL |
| Relationships as attributes | Another dependency to keep current |

> **The rule of thumb:** an ORM pays off for **application code** — lots of small, similar
> CRUD operations over a stable schema. It gets in the way for **analytics** — a few large,
> complex, hand-tuned queries. Many real systems use both, and that is fine.

### Why SQLAlchemy

It is the de-facto standard in Python, and unusually it exposes **two layers**:

- **Core** — a SQL expression language. You still think in tables and `SELECT`s, but build
  them from Python objects rather than strings.
- **ORM** — classes mapped to tables, built on top of Core.

You can drop from the ORM down to Core, or to raw SQL, at any point. That escape hatch is
the main reason it is trusted for large systems.

> **Version note:** SQLAlchemy **2.0** (2023) changed the recommended style substantially.
> Anything you find using `Query`, `session.query(...)` or `declarative_base()` is the old
> 1.x idiom. This notebook uses the 2.0 style: `select()`, `Session`, `Mapped` and
> `mapped_column`.

In [ ]:
import importlib.util

if importlib.util.find_spec("sqlalchemy") is None:
    print("SQLAlchemy is not installed.")
    print("Install it into your virtual environment (see 7.2):")
    print("    python -m pip install SQLAlchemy")
    HAS_SQLALCHEMY = False
else:
    import sqlalchemy

    HAS_SQLALCHEMY = True
    print("SQLAlchemy", sqlalchemy.__version__)
    major = int(sqlalchemy.__version__.split(".")[0])
    print("2.0-style API available:", major >= 2)
    if major < 2:
        print("  ⚠️ This notebook uses the 2.0 style; upgrade with:")
        print("     python -m pip install --upgrade SQLAlchemy")

### Defining models

A model is a normal Python class that inherits from a `DeclarativeBase` subclass. The
annotations are not decoration — SQLAlchemy **reads them** to work out the column types, in
the same way `@dataclass` does (**5.3**).

```
class Product(Base):
    __tablename__ = "product"

    id:    Mapped[int] = mapped_column(primary_key=True)
    name:  Mapped[str]                          <- NOT NULL, inferred from the annotation
    note:  Mapped[str | None]                   <- NULLABLE, inferred from `| None`
```

| Annotation | Column |
|---|---|
| `Mapped[int]` | `INTEGER NOT NULL` |
| `Mapped[str]` | `VARCHAR NOT NULL` |
| `Mapped[str \| None]` | `VARCHAR NULL` |
| `Mapped[float]` | `FLOAT NOT NULL` |
| `Mapped[list["Order"]]` | A **relationship**, not a column |

In [ ]:
from sqlalchemy import String, ForeignKey, create_engine, select, func
from sqlalchemy.orm import (DeclarativeBase, Mapped, mapped_column,
                            relationship, Session, selectinload)


class Base(DeclarativeBase):
    """Every model inherits from this."""


class Product(Base):
    __tablename__ = "product"

    id: Mapped[int] = mapped_column(primary_key=True)
    sku: Mapped[str] = mapped_column(String(20), unique=True)
    name: Mapped[str] = mapped_column(String(100))
    price: Mapped[float]
    in_stock: Mapped[int] = mapped_column(default=0)

    # NOT a column - a relationship to the other table
    orders: Mapped[list["Order"]] = relationship(back_populates="product",
                                                 cascade="all, delete-orphan")

    def __repr__(self) -> str:
        return f"Product(sku={self.sku!r}, name={self.name!r}, price={self.price})"


class Order(Base):
    __tablename__ = "customer_order"

    id: Mapped[int] = mapped_column(primary_key=True)
    product_id: Mapped[int] = mapped_column(ForeignKey("product.id"))
    quantity: Mapped[int]
    customer: Mapped[str | None] = mapped_column(String(60))     # nullable

    product: Mapped["Product"] = relationship(back_populates="orders")

    def __repr__(self) -> str:
        return f"Order(id={self.id}, qty={self.quantity}, customer={self.customer!r})"


print("models defined:", [m.class_.__name__ for m in Base.registry.mappers])
print()
for column in Product.__table__.columns:
    print(f"  Product.{column.name:<10} {str(column.type):<12} "
          f"nullable={column.nullable} pk={column.primary_key}")

### Engine and Session

Two objects, with two different jobs:

| Object | Is | Lifetime |
|---|---|---|
| **`Engine`** | The connection pool and dialect. Knows *how* to talk to the database. | One per application |
| **`Session`** | A workspace for a unit of work. Tracks the objects you have loaded and changed. | One per request / task / transaction |

The connection string (`"sqlite:///file.db"`) is the *only* line that changes when you move
to another database:

```
sqlite:///inventory.db              a file
sqlite:///:memory:                  in RAM
postgresql+psycopg://user:pw@host/db
mysql+pymysql://user:pw@host/db
```

> `echo=True` on the engine prints every statement SQLAlchemy generates. It is the single
> most useful debugging tool here — an ORM that hides the SQL is an ORM you cannot reason
> about.

In [ ]:
# In-memory database, so nothing is written to disk (see 10.3)
engine = create_engine("sqlite:///:memory:", echo=False)

# Create every table defined on Base
Base.metadata.create_all(engine)

print("tables created:", list(Base.metadata.tables))

# ---- See the SQL SQLAlchemy actually generated ----
from sqlalchemy.schema import CreateTable

print()
print(CreateTable(Product.__table__).compile(engine))

### Create: adding objects

A `Session` is a **unit of work**. You add Python objects to it, and on `commit()` it works
out the `INSERT`/`UPDATE`/`DELETE` statements needed and runs them in one transaction.

Notice you never write an `INSERT`, and you never set the `id` — the database assigns it and
SQLAlchemy reads it back onto the object.

In [ ]:
with Session(engine) as session:
    keyboard = Product(sku="KB-01", name="Mechanical Keyboard", price=1299.50, in_stock=12)
    monitor  = Product(sku="MN-27", name="27-inch Monitor",    price=12499.00, in_stock=5)
    cable    = Product(sku="CB-USB", name="USB-C Cable",         price=349.99, in_stock=40)

    print("before commit, id is:", keyboard.id)

    session.add_all([keyboard, monitor, cable])
    session.commit()

    print("after commit,  id is:", keyboard.id)
    print()
    for product in session.scalars(select(Product)):
        print(" ", product)

### Read: `select()`

SQLAlchemy 2.0 uses `select()` for every query, in both Core and the ORM.

| You want | Call |
|---|---|
| A list of objects | `session.scalars(stmt).all()` |
| One object, or `None` | `session.scalars(stmt).first()` |
| Exactly one (error otherwise) | `session.scalars(stmt).one()` |
| Rows of columns | `session.execute(stmt).all()` |

`scalars()` unwraps the single-column result so you get `Product` objects rather than
one-element rows.

> 🔴 **The filters are Python expressions, not strings.** `Product.price > 1000` builds a SQL
> expression object. That means the value is **automatically parameterised** — the injection
> problem from **10.3** cannot occur through this API.

In [ ]:
with Session(engine) as session:
    print("all products:")
    for p in session.scalars(select(Product).order_by(Product.sku)):
        print("  ", p)

    # WHERE
    stmt = select(Product).where(Product.price > 1000).order_by(Product.price.desc())
    print("\nover 1000:", session.scalars(stmt).all())

    # Several conditions
    stmt = select(Product).where(Product.price < 2000, Product.in_stock > 10)
    print("cheap and in stock:", session.scalars(stmt).all())

    # first() / one_or_none()
    one = session.scalars(select(Product).where(Product.sku == "MN-27")).one()
    print("\none():", one)
    print("first() on no match:",
          session.scalars(select(Product).where(Product.sku == "NOPE")).first())

    # Selecting COLUMNS rather than objects
    rows = session.execute(select(Product.sku, Product.price).order_by(Product.price)).all()
    print("\ncolumn rows:", rows)

    # Aggregates
    total, count = session.execute(
        select(func.sum(Product.price * Product.in_stock), func.count(Product.id))
    ).one()
    print(f"\ninventory value: {total:,.2f} across {count} products")

    # 🔴 The generated SQL is parameterised - look at the ? placeholders
    stmt = select(Product).where(Product.sku == "'; DROP TABLE product; --")
    print("\ngenerated SQL:")
    print(" ", str(stmt))
    print("injection attempt returns:", session.scalars(stmt).all())

### Update and delete

You do not write `UPDATE`. You **change the attribute**, and the session notices.

That is the unit-of-work pattern: the session compares each loaded object against its
original state, and emits only the statements needed. Deleting is `session.delete(obj)`.

In [ ]:
with Session(engine) as session:
    product = session.scalars(select(Product).where(Product.sku == "KB-01")).one()
    print("before:", product)

    product.price = 1199.00          # just assign - no UPDATE written by hand
    product.in_stock -= 2

    print("dirty objects before commit:", session.dirty)
    session.commit()
    print("after :", product)

    # ---- Bulk update, without loading the objects ----
    from sqlalchemy import update

    session.execute(update(Product).where(Product.in_stock < 10).values(in_stock=0))
    session.commit()
    for p in session.scalars(select(Product).order_by(Product.sku)):
        print(f"  {p.sku:<7} stock={p.in_stock}")

    # ---- Delete ----
    doomed = session.scalars(select(Product).where(Product.sku == "CB-USB")).one()
    session.delete(doomed)
    session.commit()
    print("\nremaining:", session.scalars(select(Product.sku)).all())

    # ---- Rollback: the session is a transaction ----
    survivor = session.scalars(select(Product).where(Product.sku == "MN-27")).one()
    survivor.price = 1.00
    print("\nprice set to:", survivor.price)
    session.rollback()
    print("after rollback:", survivor.price)

### Relationships

`relationship()` is the ORM's headline feature: a foreign key becomes an **attribute**.

```python
order.product          # the Product object - a JOIN, done for you
product.orders         # the list of Orders - the reverse
```

`back_populates` keeps both sides in sync: append to `product.orders` and the matching
`order.product` is set automatically.

In [ ]:
with Session(engine) as session:
    monitor = session.scalars(select(Product).where(Product.sku == "MN-27")).one()

    # Append to the relationship - no product_id set by hand
    monitor.orders.append(Order(quantity=2, customer="Aditya"))
    monitor.orders.append(Order(quantity=1, customer="Priya"))

    keyboard = session.scalars(select(Product).where(Product.sku == "KB-01")).one()
    keyboard.orders.append(Order(quantity=5, customer="Rahul"))

    session.commit()

    print("orders on the monitor:", monitor.orders)
    print("\nnavigating BACK from an order:")
    order = session.scalars(select(Order)).first()
    print("  order  :", order)
    print("  product:", order.product.name, "- reached by attribute, not a JOIN you wrote")

    # ---- Joins, when you want them explicitly ----
    print("\nunits ordered per product:")
    rows = session.execute(
        select(Product.sku, func.count(Order.id), func.sum(Order.quantity))
        .join(Order, isouter=True)
        .group_by(Product.id)
        .order_by(Product.sku)
    ).all()
    for sku, n_orders, units in rows:
        print(f"  {sku:<7} orders={n_orders} units={units or 0}")

### 🔴 The N+1 query problem

This is **the** performance trap of every ORM, and the main reason people say ORMs are slow.

By default a relationship is **lazy**: it is not loaded until you touch it. So this loop:

```python
for product in session.scalars(select(Product)):     # 1 query
    print(len(product.orders))                       # +1 query, per product
```

runs **1 + N** queries. With 3 products you will not notice. With 3,000 you have made 3,001
round trips to the database and the page takes twenty seconds.

### The fix: eager loading

Tell SQLAlchemy up front that you will need the relationship:

| Strategy | Emits | Best for |
|---|---|---|
| `selectinload(Product.orders)` | 2 queries (a second `IN (...)` query) | **Usually the right default** |
| `joinedload(Product.orders)` | 1 query with a `LEFT JOIN` | Small, one-to-one-ish relations |
| default (lazy) | 1 + N | Only when you will not touch the relationship |

The `echo=True` output below shows the difference directly — count the `SELECT`s.

In [ ]:
import logging

# Capture the SQL SQLAlchemy emits, so we can count queries
class QueryCounter(logging.Handler):
    def __init__(self):
        super().__init__()
        self.queries = []

    def emit(self, record):
        msg = record.getMessage().strip()
        if msg.upper().startswith("SELECT"):
            self.queries.append(msg.split("\n")[0][:60])


sql_log = logging.getLogger("sqlalchemy.engine")
sql_log.setLevel(logging.INFO)

# ---- Lazy loading: 1 + N ----
counter = QueryCounter()
sql_log.addHandler(counter)

with Session(engine) as session:
    for product in session.scalars(select(Product)):
        _ = len(product.orders)              # triggers a query EACH TIME

sql_log.removeHandler(counter)
lazy_count = len(counter.queries)
print(f"lazy loading      : {lazy_count} SELECT statements")
for q in counter.queries:
    print("   ", q)

# ---- Eager loading: 2 ----
counter = QueryCounter()
sql_log.addHandler(counter)

with Session(engine) as session:
    stmt = select(Product).options(selectinload(Product.orders))
    for product in session.scalars(stmt):
        _ = len(product.orders)              # already loaded

sql_log.removeHandler(counter)
eager_count = len(counter.queries)
print(f"\nselectinload      : {eager_count} SELECT statements")
for q in counter.queries:
    print("   ", q)

print(f"""
With {lazy_count - 1} products the difference is {lazy_count} queries vs {eager_count}.
With 3,000 products it is 3,001 vs 2.

This is why `echo=True` matters: an ORM that hides its SQL hides this too.
""")

### Dropping down to Core, and to raw SQL

The ORM is a convenience, not a prison. Two escape hatches, in increasing order of bluntness:

1. **Core** — the SQL expression language, without the object mapping. Still composable,
   still parameterised.
2. **`text()`** — literal SQL. Use it for anything the expression language cannot express
   (window functions, CTEs, vendor-specific syntax) — but **still bind parameters**, never
   f-strings.

In [ ]:
from sqlalchemy import text, insert

with engine.connect() as connection:
    # ---- Core: expression language, no ORM objects ----
    stmt = (select(Product.__table__.c.sku, Product.__table__.c.price)
            .where(Product.__table__.c.price > 1000))
    print("Core result:", connection.execute(stmt).all())

    # ---- Raw SQL, with BOUND parameters ----
    result = connection.execute(
        text("SELECT sku, name FROM product WHERE price > :floor ORDER BY price DESC"),
        {"floor": 1000},
    )
    print("\nraw SQL   :", result.all())

    # 🔴 Even here: :floor is a placeholder. Never build this with an f-string.
    hostile = "'; DROP TABLE product; --"
    safe = connection.execute(
        text("SELECT sku FROM product WHERE sku = :sku"), {"sku": hostile}
    ).all()
    print("\ninjection attempt:", safe, "- tables intact:",
          connection.execute(text(
              "SELECT name FROM sqlite_master WHERE type='table'")).scalars().all())

### So: ORM or raw SQL?

| Situation | Reach for |
|---|---|
| CRUD over a stable schema, lots of small operations | **ORM** |
| A web application's models and forms | **ORM** |
| You need to support several database engines | **ORM** or Core |
| Reporting, analytics, complex aggregation | **Raw SQL** or Core |
| Bulk loading millions of rows | **Raw SQL** / `COPY` / `executemany` |
| A one-off script against one database | **`sqlite3`** (**10.3**) — no dependency |
| You cannot afford a dependency | **`sqlite3`** / a DB-API driver |
| Query performance is the product | **Raw SQL**, tuned by hand |

Most real systems use a mixture: the ORM for the application's day-to-day objects, and hand
written SQL for the handful of queries where it matters. SQLAlchemy is designed for exactly
that, which is why it exposes both layers.

> **The one thing not to do:** use an ORM without ever looking at the SQL it generates. Turn
> `echo=True` on, read it, and you will keep the benefits without the surprises.

In [ ]:
# Tidy up
Base.metadata.drop_all(engine)
engine.dispose()
print("tables dropped and engine disposed")

---

## Common Mistakes & Pitfalls

1. 🔴 **The N+1 query problem.** Iterating objects and touching a lazy relationship issues one query per object. Use `selectinload()` / `joinedload()`.
2. **Never looking at the generated SQL.** Turn `echo=True` on while developing.
3. **Using a `Session` as a long-lived global.** One session per request, task or transaction — they are cheap.
4. **Using objects after their session closed.** Attributes not yet loaded raise `DetachedInstanceError`. Load what you need, or use `expire_on_commit=False`.
5. **Following a 1.x tutorial.** `session.query(...)` and `declarative_base()` are the old API. 2.0 uses `select()` and `DeclarativeBase`.
6. **Forgetting `commit()`.** The session buffers changes until you ask for them.
7. **Building `text()` SQL with f-strings.** The ORM parameterises for you; the moment you drop to `text()`, that is your job again.
8. **Reaching for an ORM for a 30-line script.** `sqlite3` is in the standard library and has no learning curve.
9. **Expecting the ORM to design your schema.** Indexes, normalisation and query plans are still your problem.

## Best Practices

- Use the **2.0 style**: `select()`, `Session`, `Mapped`, `mapped_column`.
- Develop with `echo=True`; turn it off in production.
- One `Session` per unit of work, opened with `with Session(engine) as session:`.
- Add `selectinload()` the moment you iterate objects and touch a relationship.
- Keep the connection string in an environment variable, never in the source (**10.2**).
- Let the ORM handle CRUD; drop to Core or `text()` for reporting queries.
- Bind parameters even in `text()`.
- Use a migration tool (**Alembic**) once the schema is real — `create_all()` is for prototypes.
- Prototype against SQLite and switch the URL to Postgres when you deploy.

## Practice Exercises

Try these before moving on.

1. Add a `Category` model and give `Product` a many-to-one relationship to it.
2. Write a query returning the three most-ordered products, using `join` and `group_by`.
3. Reproduce the N+1 problem with 50 products, then fix it and count the queries.
4. Turn on `echo=True` and read the SQL generated by one `select()` with two `where` clauses.
5. Write the same 'products over 1000' query three ways: ORM, Core, and raw `text()`.
6. Show a `DetachedInstanceError` by using an object after its session closed, then fix it.
7. Add a `UniqueConstraint` and demonstrate the `IntegrityError` it raises.
8. Take the inventory schema from **10.3** and re-implement it as SQLAlchemy models. Which version would you rather maintain, and why?